# optimizer-init-params-list — ex1: materialize a generator of params into a list at init

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `optimizer-init-params-list`. Running the final beacon cell reports progress against the `PyTorch: Optimizer init` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Optimizer init` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-init-params-list`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-init-params-list"
DD_SUBTOPIC = "PyTorch: Optimizer init"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `optim.SGD(params, lr=...)` — quick refresher

An optimizer is constructed with an iterable of `nn.Parameter` tensors. Internally PyTorch's optimizer immediately materializes that iterable into a list — but if you ROLL YOUR OWN optimizer (as the ARENA SGD exercise does), you must do it yourself: `self.params = list(params)`. The reason: a generator can only be iterated once. If you store the generator and iterate it during `.step()`, the second call hits an empty iterator and silently does nothing.

**The model.parameters() trap.** `model.parameters()` returns a generator. If your optimizer stores `self.params = params` instead of `list(params)`, the first `.step()` consumes it and every subsequent step is a no-op. Tests pass on iteration 1 and fail mysteriously on iteration 2.

### Exercise 1 — materialize a generator of params into a list at init

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze why `self.params = list(params)` is required in a hand-rolled optimizer's `__init__` and implement the fix so that the optimizer survives being passed a generator.
> Keywords: optimizer-init, generator, list-materialization
> ```

**KCs targeted:** `optimizer-init-list-vs-generator`, `optimizer-init-stores-params-attribute`

You are given a `BuggyOptimizer` whose `__init__` stores the raw `params` iterable (often a generator). On the second call to `.step()` the optimizer silently does nothing because the generator was already consumed.

Implement `Ex1FixedOptimizer.__init__(self, params, lr)` to fix the bug. The contract:

1. Materialize the iterable: `self.params = list(params)`.
2. Store `self.lr = lr`.
3. Provide a `.step()` method that does an in-place vanilla SGD update for every param that has a non-None `.grad`:
   `p.data -= self.lr * p.grad`.
4. Provide a `.zero_grad()` method that sets every `p.grad = None`.

The test verifies:
- The fixed optimizer works when given a `model.parameters()` generator (the canonical PyTorch pattern).
- The fixed optimizer's `.step()` actually mutates params on the SECOND call (the buggy one fails this).
- The buggy version is demonstrably broken on the second `.step()` for the same input.

Decorate `.step()` with `@t.no_grad()` so the in-place mutation doesn't pollute the autograd graph.

In [ ]:
class BuggyOptimizer:
    # BUG: stores generator as-is
    def __init__(self, params, lr):
        self.params = params
        self.lr = lr

    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None


class Ex1FixedOptimizer:
    def __init__(self, params, lr):
        self.params = list(params)   # <-- the critical fix
        self.lr = lr

    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None


<details><summary>Solution</summary>

```python
class BuggyOptimizer:
    # BUG: stores generator as-is
    def __init__(self, params, lr):
        self.params = params
        self.lr = lr

    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None


class Ex1FixedOptimizer:
    def __init__(self, params, lr):
        self.params = list(params)   # <-- the critical fix
        self.lr = lr

    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None
```

**Why this bug is so insidious.** It survives one full step. Loss does decrease on iteration 1. The optimizer 'works' for exactly one batch. Tests that use a single-step smoke check will pass. Then every subsequent step is a silent no-op and the loss curve flatlines.

**PyTorch's built-in optimizers already do this for you.** `torch.optim.SGD.__init__` calls `param_groups = list(params)` internally — that's why you never see this bug with the official API. The trap only matters for hand-rolled optimizers, which is exactly what ARENA chapter 0 part 3 asks you to write.

**General Python lesson.** Any constructor that takes an `Iterable[T]` and intends to iterate it MORE THAN ONCE must materialize it: `list(it)`, `tuple(it)`, or `dict(it)`. The type annotation `Iterable` is the warning sign.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()